# 04 - Random Forest baseline on BGL windows

> **Note: this is a small pipeline test, not a final result.** The BGL 2k sample gives only 100 windows (30 in the test set), so one misclassified window moves the scores by several points, and feature importances / SHAP values are learned from just 70 windows. The point is to check that the supervised pipeline - windows → classifier → metrics → explanation - works end to end. Real results need the full BGL dataset.

Setup:

- `data/processed/bgl_2k_windows.csv`: 100 windows of 20 lines, one event-count column per Drain template, `label` = 1 if any line is an alert
- Same split as notebook 03: first 70% of windows (in time order) for training, last 30% for testing
- Unlike notebook 03 this model is **supervised**: it trains on both normal and anomaly windows of the training part
- `class_weight="balanced"` so the rarer anomaly class weighs as much as the normal class

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import shap
from sklearn.ensemble import RandomForestClassifier

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.evaluation.metrics import evaluate

pd.set_option("display.max_colwidth", 100)

SEED = 42
TRAIN_FRAC = 0.7

data = pd.read_csv(ROOT / "data" / "processed" / "bgl_2k_windows.csv", index_col="window")
templates = pd.read_csv(ROOT / "data" / "processed" / "bgl_2k_templates.csv").set_index("column")["template"]

X, y = data.drop(columns="label"), data["label"]
n_train = int(len(data) * TRAIN_FRAC)
X_train, y_train = X.iloc[:n_train], y.iloc[:n_train]
X_test, y_test = X.iloc[n_train:], y.iloc[n_train:]

pd.DataFrame({
    "windows": [len(X_train), len(X_test)],
    "anomaly windows": [y_train.sum(), y_test.sum()],
}, index=["train (first 70%)", "test (last 30%)"])

,windows,anomaly windows
train (first 70%),70,15
test (last 30%),30,13


## Train and evaluate

In [2]:
rf = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=SEED, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
proba = rf.predict_proba(X_test)[:, 1]

results = pd.DataFrame([{"model": "Random Forest (balanced)", **evaluate(y_test, y_pred)}]).set_index("model")
results.style.format({c: "{:.3f}" for c in ["precision", "recall", "f1", "false_alarm_rate"]})

,precision,recall,f1,false_alarm_rate,tp,fp,tn,fn
model,,,,,,,,
Random Forest (balanced),0.750,0.462,0.571,0.118,6,2,15,7


In [3]:
pd.DataFrame({
    "true label": y_test,
    "predicted": y_pred,
    "P(anomaly)": proba.round(2),
}).T

window,70,71,72,73,74,75,76,77,78,79,...,90,91,92,93,94,95,96,97,98,99
true label,1.00,0.00,0.00,1.00,1.00,1.00,1.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.0,1.00
predicted,0.00,0.00,1.00,0.00,1.00,1.00,1.00,0.00,0.00,0.00,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.0,0.00
P(anomaly),0.41,0.35,0.68,0.44,0.65,0.68,0.65,0.26,0.27,0.27,...,0.55,0.36,0.36,0.36,0.46,0.36,0.34,0.53,0.6,0.31


## Top 10 feature importances

Impurity-based importance: how much each template's count helps the trees separate normal from anomaly windows, averaged over all trees.

In [4]:
importances = (
    pd.DataFrame({"template_id": X.columns, "importance": rf.feature_importances_})
    .assign(template=lambda d: d["template_id"].map(templates))
    .sort_values("importance", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
importances.index += 1
importances.style.format({"importance": "{:.3f}"})

,template_id,importance,template
1,E5,0.170,generating core.
2,E10,0.119,data TLB error interrupt
3,E3,0.072,"CE sym , at , mask"
4,E13,0.056,instruction address:
5,E4,0.050,ciod: failed to read message prefix on control stream (CioStream socket to
6,E43,0.046,"ciod: Error loading <*> invalid or missing program image, No such file or directory"
7,E12,0.046,data storage interrupt
8,E2,0.043,double-hummer alignment exceptions
9,E1,0.041,instruction cache parity error corrected
10,E55,0.036,floating point alignment exceptions


## Explaining one detected anomaly with SHAP

SHAP splits the model's predicted anomaly probability for one window into a contribution per template: starting from the average prediction, each template's count pushes the probability up or down. We pick the correctly detected anomaly window the model is most confident about.

In [5]:
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)
# Depending on the shap version this is a list per class or an array (windows, features, classes)
shap_anomaly = shap_values[1] if isinstance(shap_values, list) else shap_values[..., 1]
base_value = np.atleast_1d(explainer.expected_value)[-1]

detected = np.where((y_test.to_numpy() == 1) & (y_pred == 1))[0]
pos = detected[np.argmax(proba[detected])]
window = X_test.index[pos]

contrib = (
    pd.DataFrame({
        "template_id": X.columns,
        "count_in_window": X_test.iloc[pos].to_numpy(),
        "shap": shap_anomaly[pos],
    })
    .assign(template=lambda d: d["template_id"].map(templates))
    .sort_values("shap", ascending=False)
)
top3 = contrib.head(3)
top3

,template_id,count_in_window,shap,template
42,E43,1,0.141001,"ciod: Error loading <*> invalid or missing program image, No such file or directory"
4,E5,0,0.087669,generating core.<NUM>
2,E3,2,0.064839,"CE sym <NUM>, at <HEX>, mask <HEX>"


In [6]:
def explain(window, prob, base, top):
    lines = [
        f"Window {window} (log lines {window * 20 + 1}-{window * 20 + 20}) was flagged as an anomaly "
        f"with probability {prob:.0%} (the average window scores {base:.0%}). The three events that pushed it up most:",
        "",
    ]
    for i, row in enumerate(top.itertuples(), start=1):
        seen = (
            f"appeared {row.count_in_window} time{'s' if row.count_in_window != 1 else ''}"
            if row.count_in_window > 0
            else "did not appear (in training, windows without it were more often anomalies)"
        )
        lines.append(f'{i}. "{row.template}" ({row.template_id}) {seen} - raising the anomaly probability by {row.shap:+.0%}.')
    return "\n".join(lines)


print(explain(window, proba[pos], base_value, top3))

Window 75 (log lines 1501-1520) was flagged as an anomaly with probability 68% (the average window scores 50%). The three events that pushed it up most:

1. "ciod: Error loading <*> invalid or missing program image, No such file or directory" (E43) appeared 1 time - raising the anomaly probability by +14%.
2. "generating core.<NUM>" (E5) did not appear (in training, windows without it were more often anomalies) - raising the anomaly probability by +9%.
3. "CE sym <NUM>, at <HEX>, mask <HEX>" (E3) appeared 2 times - raising the anomaly probability by +6%.


**Reminder:** this is a small pipeline test. With 70 training windows, importances and SHAP values reflect this tiny sample, and the test metrics are very noisy. The same code will be re-run on the full BGL data.